# Classification of Cardiac Arrhythmias from 2D ECG Images in a Remote Monitoring System

This notebook runs the image generation, CNN training and evaluation workflow with modular Python code.
The [academic report](../docs/academic-report-it.pdf) documents the university examination's results, which should not be interpreted as results from this notebook.
For context, methodology, dataset attribution, experimental scope and class definitions, read the repository [README](../README.md).

## 1. Environment and data

From the repository root, install `pip install -e ".[notebook,test]"` and run Jupyter here.
Alternatively, prepare the runtime and import the package before running the following cells.
The official MIT-BIH recordings are downloaded only if the dataset directory does not exist. Both downloaded and manually supplied record triplets are checked against pinned SHA-256 hashes before preprocessing.

In [ ]:
from pathlib import Path
import numpy as np

from ecg_classification.data import (
    CLASS_NAMES, download_mit_bih, generate_record_images,
    load_images, record_ids, record_class_labels, split_records, split_class_coverage,
    verify_mit_bih_records,
)
from ecg_classification.model import train_model, predict_model
from ecg_classification.evaluation import evaluate_predictions

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    if (PROJECT_ROOT.parent / "pyproject.toml").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise RuntimeError("Run this notebook from the repository root or notebooks/ directory")
DATASET_DIRECTORY = PROJECT_ROOT / "mit-bih-arrhythmia-database-1.0.0"
IMAGE_DIRECTORY = PROJECT_ROOT / "dataset_images"
CHECKPOINT = PROJECT_ROOT / "cnn.h5"

# Small-scale walkthrough; set None and 30 respectively for a full run.
MAX_BEATS_PER_RECORD = 5
EPOCHS = 1
# Enable nine-crop augmentation only for training when requested.
INCLUDE_TRAIN_CROPS = False
SEED = 0


In [ ]:
if not DATASET_DIRECTORY.is_dir():
    download_mit_bih(DATASET_DIRECTORY)
verify_mit_bih_records(DATASET_DIRECTORY)
records = record_ids(DATASET_DIRECTORY)
assert len(records) == 48, "Expected all 48 primary MIT-BIH records"
record_labels = record_class_labels(DATASET_DIRECTORY, records)
partitions = split_records(records, seed=SEED, record_labels=record_labels)
print({name: len(items) for name, items in partitions.items()})
print("Classes present in each partition:", split_class_coverage(partitions, record_labels))


## 2. ECG segmentation and image creation

A segment extends from the *previous* annotation + 20 samples to the *next* annotation - 20 samples, using ECG channel 0 and Matplotlib rendering. Each accepted segment yields one 128 x 128 grayscale PNG. Nine cropped variants are generated only for training when `INCLUDE_TRAIN_CROPS=True`. This may take significant time and disk space with `MAX_BEATS_PER_RECORD = None`. The three partitions have disjoint record IDs and, where feasible, contain examples of the available classes. Single-record classes can appear in only one partition.

In [ ]:
generated = {}
for partition, identifiers in partitions.items():
    count = 0
    for record in identifiers:
        count += generate_record_images(
            DATASET_DIRECTORY,
            IMAGE_DIRECTORY,
            record,
            partition,
            augmentation=INCLUDE_TRAIN_CROPS and partition == "train",
            maximum_beats=MAX_BEATS_PER_RECORD,
        )
    generated[partition] = count
print("Base ECG images generated:", generated)


## 3. Build the training, validation and test inputs

By default only base images with suffix `0.png` are loaded. Augmentation can also be applied online to training images. `INCLUDE_TRAIN_CROPS=True` adds crop variants to the training distribution.

In [ ]:
train_images = load_images(
    IMAGE_DIRECTORY, "train", partitions["train"], include_crops=INCLUDE_TRAIN_CROPS
)
validation_images = load_images(IMAGE_DIRECTORY, "validation", partitions["validation"])
test_images = load_images(IMAGE_DIRECTORY, "test", partitions["test"])
print("Model input images:", len(train_images), len(validation_images), len(test_images))
assert train_images and validation_images and test_images, "A nonempty split is required"
assert not ({sample.record for sample in train_images} & {sample.record for sample in test_images})
assert not ({sample.record for sample in train_images} & {sample.record for sample in validation_images})
assert not ({sample.record for sample in validation_images} & {sample.record for sample in test_images})


## 4. CNN training

The model uses: six convolutional layers, eight-class softmax, batch size 32, Adam, and categorical crossentropy. The notebook defaults to a **single-epoch functionality check**, not scientific performance evaluation. The best validation-loss checkpoint is saved locally to `cnn.h5`; model weights are not part of the published repository.

In [ ]:
trained_model = train_model(
    train_images, validation_images,
    epochs=EPOCHS, checkpoint=CHECKPOINT,
    batch_size=32, augmentation=True, seed=SEED,
)


## 5. Evaluation

Test images are evaluated **without random augmentation**; the confusion matrix uses ground truth as rows and predicted classes as columns. These results correspond to the model trained by this execution. Reported per-class support identifies classes absent from test; macro averages include all eight output indices and do not establish performance for zero-support classes.

In [ ]:
from tensorflow.keras.models import load_model

best_model = load_model(CHECKPOINT)
predictions = predict_model(best_model, test_images)
results = evaluate_predictions(predictions, test_images)
print(f"Weighted precision: {results['precision']:.4f}")
print(f"Weighted recall:    {results['recall']:.4f}")
print(f"Weighted F1 score:  {results['f1']:.4f}")
print(f"Macro F1 (all 8 classes): {results['macro_f1']:.4f}")
print(f"Accuracy:      {results['accuracy']:.4f}")
print("Per-class metrics (support=0 means not evaluated in this test split):")
for label, metrics in results["per_class"].items():
    print(f"  {label:5s} support={metrics['support']:5d} "
          f"precision={metrics['precision']:.4f} "
          f"recall={metrics['recall']:.4f} f1={metrics['f1']:.4f}")
print("Class ordering:", CLASS_NAMES)
print("Confusion matrix (true rows, predicted columns):")
print(results["confusion_matrix"])
